In [ ]:
#@title Install requirements. { display-mode: "form" }
# Install requirements
#Da arquitetura prot5
!pip install torch transformers sentencepiece h5py

In [ ]:
import sys
print(sys.version)

3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU disponível:", torch.cuda.get_device_name(0))
else:
    print("GPU não está disponível")

GPU disponível: NVIDIA L4


In [ ]:
#@title Set up working directories and download files/checkpoints. { display-mode: "form" }
# Create directory for storing model weights (2.3GB) and example sequences.
# Here we use the encoder-part of ProtT5-XL-U50 in half-precision (fp16) as
# it performed best in our benchmarks (also outperforming ProtBERT-BFD).
# Also download secondary structure prediction checkpoint to show annotation extraction from embeddings
!mkdir protT5 # root directory for storing checkpoints, results etc
!mkdir protT5/protT5_checkpoint # directory holding the ProtT5 checkpoint
!mkdir protT5/sec_struct_checkpoint # directory storing the supervised classifier's checkpoint
!mkdir protT5/output # directory for storing your embeddings & predictions
#!wget -nc -P protT5/ https://rostlab.org/~deepppi/example_seqs.fasta
# Huge kudos to the bio_embeddings team here! We will integrate the new encoder, half-prec ProtT5 checkpoint soon
!wget -nc -P protT5/sec_struct_checkpoint http://data.bioembeddings.com/public/embeddings/feature_models/t5/secstruct_checkpoint.pt

--2025-12-10 03:07:51--  http://data.bioembeddings.com/public/embeddings/feature_models/t5/secstruct_checkpoint.pt
Resolving data.bioembeddings.com (data.bioembeddings.com)... 143.95.108.236
Connecting to data.bioembeddings.com (data.bioembeddings.com)|143.95.108.236|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3727951 (3.6M)
Saving to: ‘protT5/sec_struct_checkpoint/secstruct_checkpoint.pt’

secstruct_checkpoin 100%[===================>]   3.55M  --.-KB/s    in 0.1s    

2025-12-10 03:07:51 (26.1 MB/s) - ‘protT5/sec_struct_checkpoint/secstruct_checkpoint.pt’ saved [3727951/3727951]



In [ ]:
# In the following you can define your desired output. Current options:
# per_residue embeddings
# per_protein embeddings
# secondary structure predictions

# Replace this file with your own (multi-)FASTA
# Headers are expected to start with ">";
seq_path = "./protT5/example_seqs.fasta"

# whether to retrieve embeddings for each residue in a protein
# --> Lx1024 matrix per protein with L being the protein's length
# as a rule of thumb: 1k proteins require around 1GB RAM/disk
per_residue = False
per_residue_path = "./protT5/output/per_residue_embeddings.h5" # where to store the embeddings

# whether to retrieve per-protein embeddings
# --> only one 1024-d vector per protein, irrespective of its length
per_protein = True
per_protein_path = "./protT5/output/per_protein_embeddings.h5" # where to store the embeddings

# whether to retrieve secondary structure predictions
# This can be replaced by your method after being trained on ProtT5 embeddings
sec_struct = True
sec_struct_path = "./protT5/output/ss3_preds.fasta" # file for storing predictions

# make sure that either per-residue or per-protein embeddings are stored
assert per_protein is True or per_residue is True or sec_struct is True, print(
    "Minimally, you need to active per_residue, per_protein or sec_struct. (or any combination)")


In [ ]:
!pip install openpyxl

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Mount your Google Drive
from google.colab import drive
import pandas as pd

######### Alterar o arquivo sempre que for realizar novo embedding #########
#Mudar nome do arquivo a ser salvo tambem

# Define the path to your dataset
data_path = "/content/drive/MyDrive/TCC_Gabriel/Scripts/Artigo/bcell_FINAL_22AA_09_12.xlsx"

# Read the dataset using pandas
df_data_all = pd.read_excel(data_path)

# Your data is now loaded into the dataframe 'df'
df_data_all


,Assay Antigen - Name,Assay - Method,Assay - Qualitative Measure,Assay Antigen - Source Organism,Assay Antigen - Species,Tamanho_AA
0,AAAAIAWLLGSSTSQ,microarray,"Positive, Positive-Low",Zika virus,Zika virus,15
1,AAAARAAQKRTAAGI,microarray,"Positive, Positive-Low",Zika virus,Zika virus,15
2,AAAGIMKNPTVDGIT,microarray,Positive,Dengue virus,Dengue virus,15
3,AAAGLVGVLAGLAFQ,microarray,Positive,Yellow fever virus,Yellow fever virus,15
4,AAAIAWLLGSSTSQKVIY,ELISA,Negative,Zika virus ZIKV/H. sapiens/FrenchPolynesia/100...,Zika virus,18
...,...,...,...,...,...,...
8921,YYCGGLKNVKEVKGL,microarray,Negative,Dengue virus,Dengue virus,15
8922,YYCGGLKNVREVKGL,microarray,"Positive, Positive-Low",Dengue virus,Dengue virus,15
8923,YYMATLKNVTEVKGY,microarray,Negative,Dengue virus,Dengue virus,15
8924,YYSEPTSEDNAHHVC,microarray,Positive,Yellow fever virus,Yellow fever virus,15


In [ ]:
len(df_data_all['Assay Antigen - Name'].tolist())

8926

In [ ]:
#@title Import dependencies and check whether GPU is available. { display-mode: "form" }
from transformers import T5EncoderModel, T5Tokenizer
import torch
import h5py
import time
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print("Using {}".format(device))

Using cuda:0


In [ ]:
#@title Network architecture for secondary structure prediction. { display-mode: "form" }
# Convolutional neural network (two convolutional layers) to predict secondary structure
class ConvNet( torch.nn.Module ):
    def __init__( self ):
        super(ConvNet, self).__init__()
        # This is only called "elmo_feature_extractor" for historic reason
        # CNN weights are trained on ProtT5 embeddings
        self.elmo_feature_extractor = torch.nn.Sequential(
                        torch.nn.Conv2d( 1024, 32, kernel_size=(7,1), padding=(3,0) ), # 7x32
                        torch.nn.ReLU(),
                        torch.nn.Dropout( 0.25 ),
                        )
        n_final_in = 32
        self.dssp3_classifier = torch.nn.Sequential(
                        torch.nn.Conv2d( n_final_in, 3, kernel_size=(7,1), padding=(3,0)) # 7
                        )

        self.dssp8_classifier = torch.nn.Sequential(
                        torch.nn.Conv2d( n_final_in, 8, kernel_size=(7,1), padding=(3,0))
                        )
        self.diso_classifier = torch.nn.Sequential(
                        torch.nn.Conv2d( n_final_in, 2, kernel_size=(7,1), padding=(3,0))
                        )


    def forward( self, x):
        # IN: X = (B x L x F); OUT: (B x F x L, 1)
        x = x.permute(0,2,1).unsqueeze(dim=-1)
        x         = self.elmo_feature_extractor(x) # OUT: (B x 32 x L x 1)
        d3_Yhat   = self.dssp3_classifier( x ).squeeze(dim=-1).permute(0,2,1) # OUT: (B x L x 3)
        d8_Yhat   = self.dssp8_classifier( x ).squeeze(dim=-1).permute(0,2,1) # OUT: (B x L x 8)
        diso_Yhat = self.diso_classifier(  x ).squeeze(dim=-1).permute(0,2,1) # OUT: (B x L x 2)
        return d3_Yhat, d8_Yhat, diso_Yhat

In [ ]:
#@title Load the checkpoint for secondary structure prediction. { display-mode: "form" }
def load_sec_struct_model():
  checkpoint_dir="./protT5/sec_struct_checkpoint/secstruct_checkpoint.pt"
  state = torch.load( checkpoint_dir )
  model = ConvNet()
  model.load_state_dict(state['state_dict'])
  model = model.eval()
  model = model.to(device)
  print('Loaded sec. struct. model from epoch: {:.1f}'.format(state['epoch']))

  return model

In [ ]:
#@title Load encoder-part of ProtT5 in half-precision. { display-mode: "form" }
# Load ProtT5 in half-precision (more specifically: the encoder-part of ProtT5-XL-U50)
def get_T5_model():
    model = T5EncoderModel.from_pretrained("Rostlab/prot_t5_xl_half_uniref50-enc")
    model = model.to(device) # move model to GPU
    model = model.eval() # set model to evaluation model
    tokenizer = T5Tokenizer.from_pretrained('Rostlab/prot_t5_xl_half_uniref50-enc', do_lower_case=False)

    return model, tokenizer

In [ ]:
#@title Read in file in fasta format. { display-mode: "form" }
def read_fasta( fasta_path, split_char="!", id_field=0):
    '''
        Reads in fasta file containing multiple sequences.
        Split_char and id_field allow to control identifier extraction from header.
        E.g.: set split_char="|" and id_field=1 for SwissProt/UniProt Headers.
        Returns dictionary holding multiple sequences or only single
        sequence, depending on input file.
    '''

    seqs = dict()
    with open( fasta_path, 'r' ) as fasta_f:
        for line in fasta_f:
            # get uniprot ID from header and create new entry
            if line.startswith('>'):
                uniprot_id = line.replace('>', '').strip().split(split_char)[id_field]
                # replace tokens that are mis-interpreted when loading h5
                uniprot_id = uniprot_id.replace("/","_").replace(".","_")
                seqs[ uniprot_id ] = ''
            else:
                # repl. all whie-space chars and join seqs spanning multiple lines, drop gaps and cast to upper-case
                seq= ''.join( line.split() ).upper().replace("-","")
                # repl. all non-standard AAs and map them to unknown/X
                seq = seq.replace('U','X').replace('Z','X').replace('O','X')
                seqs[ uniprot_id ] += seq
    example_id=next(iter(seqs))
    print("Read {} sequences.".format(len(seqs)))
    print("Example:\n{}\n{}".format(example_id,seqs[example_id]))

    return seqs

In [ ]:
#@title Generate embeddings. { display-mode: "form" }
# Generate embeddings via batch-processing
# per_residue indicates that embeddings for each residue in a protein should be returned.
# per_protein indicates that embeddings for a whole protein should be returned (average-pooling)
# max_residues gives the upper limit of residues within one batch
# max_seq_len gives the upper sequences length for applying batch-processing
# max_batch gives the upper number of sequences per batch
def get_embeddings( model, tokenizer, seqs, per_residue, per_protein,
                   max_residues=4000, max_seq_len=1000, max_batch=100 ):
    results = {"residue_embs" : dict(),
               "protein_embs" : dict()}

    # sort sequences according to length (reduces unnecessary padding --> speeds up embedding)
    seq_dict   = sorted( seqs.items(), key=lambda kv: len( seqs[kv[0]] ), reverse=True )
    start = time.time()
    batch = list()
    for seq_idx, (pdb_id, seq) in enumerate(seq_dict,1):
        seq = seq
        seq_len = len(seq)
        seq = ' '.join(list(seq))
        batch.append((pdb_id,seq,seq_len))

        # count residues in current batch and add the last sequence length to
        # avoid that batches with (n_res_batch > max_residues) get processed
        n_res_batch = sum([ s_len for  _, _, s_len in batch ]) + seq_len
        if len(batch) >= max_batch or n_res_batch>=max_residues or seq_idx==len(seq_dict) or seq_len>max_seq_len:
            pdb_ids, seqs, seq_lens = zip(*batch)
            batch = list()

            # add_special_tokens adds extra token at the end of each sequence
            token_encoding = tokenizer.batch_encode_plus(seqs, add_special_tokens=True, padding="longest")
            input_ids      = torch.tensor(token_encoding['input_ids']).to(device)
            attention_mask = torch.tensor(token_encoding['attention_mask']).to(device)

            try:
                with torch.no_grad():
                    # returns: ( batch-size x max_seq_len_in_minibatch x embedding_dim )
                    embedding_repr = model(input_ids, attention_mask=attention_mask)
            except RuntimeError:
                print("RuntimeError during embedding for {} (L={})".format(pdb_id, seq_len))
                continue

            for batch_idx, identifier in enumerate(pdb_ids): # for each protein in the current mini-batch
                s_len = seq_lens[batch_idx]
                # slice off padding --> batch-size x seq_len x embedding_dim
                emb = embedding_repr.last_hidden_state[batch_idx,:s_len]
                if per_residue: # store per-residue embeddings (Lx1024)
                    results["residue_embs"][ identifier ] = emb.detach().cpu().numpy().squeeze()
                if per_protein: # apply average-pooling to derive per-protein embeddings (1024-d)
                    protein_emb = emb.mean(dim=0)
                    results["protein_embs"][identifier] = protein_emb.detach().cpu().numpy().squeeze()


    passed_time=time.time()-start
    avg_time = passed_time/len(results["residue_embs"]) if per_residue else passed_time/len(results["protein_embs"])
    print('\n############# EMBEDDING STATS #############')
    print('Total number of per-residue embeddings: {}'.format(len(results["residue_embs"])))
    print('Total number of per-protein embeddings: {}'.format(len(results["protein_embs"])))
    print("Time for generating embeddings: {:.1f}[m] ({:.3f}[s/protein])".format(
        passed_time/60, avg_time ))
    print('\n############# END #############')
    return results

In [ ]:
#@title Write embeddings to disk. { display-mode: "form" }
def save_embeddings(emb_dict,out_path):
    with h5py.File(str(out_path), "w") as hf:
        for sequence_id, embedding in emb_dict.items():
            # noinspection PyUnboundLocalVariable
            hf.create_dataset(sequence_id, data=embedding)
    return None

In [ ]:
#@title Write predictions to disk. { display-mode: "form" }
def write_prediction_fasta(predictions, out_path):
  class_mapping = {0:"H",1:"E",2:"L"}
  with open(out_path, 'w+') as out_f:
      out_f.write( '\n'.join(
          [ ">{}\n{}".format(
              seq_id, ''.join( [class_mapping[j] for j in yhat] ))
          for seq_id, yhat in predictions.items()
          ]
            ) )
  return None

In [ ]:
#seq_dict_train = dict(zip(train_df['Gene_Name'].to_list(), train_df.Sequence))
#seq_dict_valid = dict(zip(val_df['Gene_Name'].to_list(), val_df.Sequence))
#seq_dict_test = dict(zip(test_df['Gene_Name'].to_list(), test_df.Sequence))

seq_dict_space = dict(zip(df_data_all['Assay Antigen - Name'].to_list(), df_data_all['Assay Antigen - Name'].to_list()))

In [ ]:
len(seq_dict_space)

8926

In [ ]:
import numpy as np
import os
import pandas as pd # Import pandas to save as CSV
'''
X_train = np.vstack(list(Embedding_train['protein_embs'].values()))
X_valid = np.vstack(list(Embedding_valid['protein_embs'].values()))
X_test = np.vstack(list(Embedding_test['protein_embs'].values()))

print(X_train.shape)
print(X_valid.shape)
print(X_test.shape)'''

#Rodando apenas o embeding

model, tokenizer = get_T5_model()

start = time.time()
Embedding_space = get_embeddings(model, tokenizer, seq_dict_space, per_residue, per_protein)
end = time.time()
print('Execution time: {} seconds'.format(end - start))

Embedding_space["protein_embs"] = {x: Embedding_space["protein_embs"][x] for x in df_data_all['Assay Antigen - Name'].to_list()}

# Define the Google Drive path to save the embeddings
#################### Trocar o nome do arquivo sempre que for realizar novo embedding ################################
google_drive_path = '/content/drive/MyDrive/TCC_Gabriel/Scripts/Artigo/embeddings_variados_09_12_25.csv' # Changed extension to .csv

# Create the directory if it doesn't exist
output_dir = os.path.dirname(google_drive_path)
os.makedirs(output_dir, exist_ok=True)

# Save the embeddings to CSV
# Convert the dictionary to a pandas DataFrame for easier saving
df_embeddings = pd.DataFrame.from_dict(Embedding_space["protein_embs"], orient='index')
df_embeddings.index.name = 'Assay Antigen - Name' # Optional: name the index column
df_embeddings.to_csv(google_drive_path) # Save to CSV

# save_embeddings(Embedding_space["protein_embs"], google_drive_path) # Removed the old save function

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/656 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.42G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.42G [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/238k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565



############# EMBEDDING STATS #############
Total number of per-residue embeddings: 0
Total number of per-protein embeddings: 8926
Time for generating embeddings: 0.7[m] (0.005[s/protein])

############# END #############
Execution time: 42.901127338409424 seconds


In [ ]:
X_space = np.vstack(list(Embedding_space['protein_embs'].values()))

In [ ]:
X_space.shape

(8926, 1024)